In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
print(documents[15])

{'id': 'd3f485cd10', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Homework: What are homework and project deadlines?', 'answer': "Deadlines are published per cohort. Find them in:\n\n- The current cohort's folder in the [course repo](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/cohorts).\n- The course website (link is in the cohort folder's README).\n- The course Google Calendar (also linked from the cohort folder).\n\nAlso keep an eye on `@Au-Tomator` posts in Slack for extension announcements. The submission form may show an updated deadline if instructors have changed it."}


In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
response.output_parsed.questions

['Is it too late to join the course if I just found it now?',
 'Can I still start the course after it has already begun?',
 'If I join late, can I still get a certificate somehow?',
 'Do I have to submit the project while submissions are open to get the certificate?',
 'What happens if I take the course now but miss the project submission deadline?']

In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late?', 'If I start the course now, can I still get a certificate?', 'What do I need to do to be eligible for the course certificate?', 'Is it too late to submit the project for certification?', 'Can I join now and still finish with a certificate if I submit the project on time?']


In [15]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the course certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to submit the project for certification?',
  'document': '74eb249bbf'},
 {'question': 'Can I join now and still finish with a certificate if I submit the project on time?',
  'document': '74eb249bbf'}]

In [16]:
import pandas as pd

In [17]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,"If I start the course now, can I still get a c...",74eb249bbf
2,What do I need to do to be eligible for the co...,74eb249bbf
3,Is it too late to submit the project for certi...,74eb249bbf
4,Can I join now and still finish with a certifi...,74eb249bbf


In [18]:
from evaluation_utils import llm_structured_retry

In [19]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [20]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [24]:
df_ground_truth = pd.DataFrame(ground_truth)

In [25]:
df_ground_truth

,question,document
0,I just found this course — can I still sign up...,74eb249bbf
1,Am I allowed to join the course after it alrea...,74eb249bbf
2,"If I enroll late, do I still get access to eve...",74eb249bbf
3,Can I still get a certificate if I join the co...,74eb249bbf
4,What do I need to do to make sure I’m eligible...,74eb249bbf
...,...,...
510,Why does pip keep giving me requests 2.28 when...,4b30b918bc
511,Is there a way to install requests straight fr...,4b30b918bc
512,I’m hitting a 401 Client Error while setting t...,4b30b918bc
513,What’s the exact pip command to install the ri...,4b30b918bc


In [26]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)